In [1]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import cv2
import os
import random

In [2]:
# Define directories
base_dir = r'C:\Users\vaibh\OneDrive\Desktop\Vaibhav_project\Vaibhav_project\CNN_dataset'  # Update this path according to your dataset
categories = ['healthy', 'iron_deficient', 'kidney_disease', 'melanoma']

# Image dimensions
IMG_SIZE = 128

def load_images(base_dir, categories, img_size):
    data = []
    for category in categories:
        path = os.path.join(base_dir, category)
        class_num = categories.index(category)  # Label assignment
        for img in os.listdir(path):
            try:
                img_array = cv2.imread(os.path.join(path, img), cv2.IMREAD_COLOR)
                resized_img = cv2.resize(img_array, (img_size, img_size))
                data.append([resized_img, class_num])
            except Exception as e:
                pass
    return data

data = load_images(base_dir, categories, IMG_SIZE)

# Shuffle data
random.shuffle(data)

# Prepare features (X) and labels (y)
X = []
y = []

for features, label in data:
    X.append(features)
    y.append(label)

X = np.array(X).reshape(-1, IMG_SIZE, IMG_SIZE, 3) / 255.0  # Normalize
y = np.array(y)

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [3]:
datagen = ImageDataGenerator(
    rotation_range=20,
    zoom_range=0.2,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True
)

datagen.fit(X_train)


In [4]:
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(IMG_SIZE, IMG_SIZE, 3)),
    MaxPooling2D(pool_size=(2, 2)),
    BatchNormalization(),

    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),
    BatchNormalization(),

    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),
    BatchNormalization(),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(64, activation='relu'),
    Dropout(0.5),
    Dense(4, activation='softmax')  # 4 classes: Healthy, Iron Deficient, Kidney Disease, Melanoma
])


C:\Users\vaibh\AppData\Roaming\Python\Python312\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [5]:
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [ ]:
history = model.fit(datagen.flow(X_train, y_train, batch_size=32), validation_data=(X_test, y_test), epochs=60)

C:\Users\vaibh\AppData\Roaming\Python\Python312\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/60
22/22 ━━━━━━━━━━━━━━━━━━━━ 9s 236ms/step - accuracy: 0.5779 - loss: 3.3579 - val_accuracy: 0.1092 - val_loss: 2.6076
Epoch 2/60
22/22 ━━━━━━━━━━━━━━━━━━━━ 5s 217ms/step - accuracy: 0.7420 - loss: 2.4399 - val_accuracy: 0.1724 - val_loss: 2.7424
Epoch 3/60
22/22 ━━━━━━━━━━━━━━━━━━━━ 5s 217ms/step - accuracy: 0.7854 - loss: 2.2512 - val_accuracy: 0.4195 - val_loss: 3.2404
Epoch 4/60
22/22 ━━━━━━━━━━━━━━━━━━━━ 5s 219ms/step - accuracy: 0.7804 - loss: 1.7771 - val_accuracy: 0.1092 - val_loss: 5.9299
Epoch 5/60
22/22 ━━━━━━━━━━━━━━━━━━━━ 5s 224ms/step - accuracy: 0.8027 - loss: 1.2819 - val_accuracy: 0.2759 - val_loss: 3.5247
Epoch 6/60
22/22 ━━━━━━━━━━━━━━━━━━━━ 5s 218ms/step - accuracy: 0.8223 - loss: 1.3070 - val_accuracy: 0.1839 - val_loss: 3.3414
Epoch 7/60
22/22 ━━━━━━━━━━━━━━━━━━━━ 5s 234ms/step - accuracy: 0.7870 - loss: 1.1047 - val_accuracy: 0.3161 - val_loss: 1.7982
Epoch 8/60
22/22 ━━━━━━━━━━━━━━━━━━━━ 5s 231ms/step - accuracy: 0.8389 - loss: 0.6795 - val_accuracy: 0.

In [ ]:
# Make predictions on test set
y_pred = np.argmax(model.predict(X_test), axis=-1)

# Print classification report
print("Classification Report:\n", classification_report(y_test, y_pred, target_names=categories))

# Plot confusion matrix
conf_matrix = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
plt.imshow(conf_matrix, cmap='Blues')
plt.title("Confusion Matrix")
plt.colorbar()
plt.xticks(np.arange(len(categories)), categories, rotation=45)
plt.yticks(np.arange(len(categories)), categories)
plt.show()


In [ ]:
# Save the trained model to a file
model.save('nail_classification_model_v3.h5')

## Below code need to run while testing the model

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import cv2
import os
import random

In [ ]:
from tensorflow.keras.models import load_model

loaded_model = load_model('nail_classification_model_v3.h5')

In [ ]:
categories = ['healthy', 'iron_deficient', 'kidney_disease', 'melanoma']

IMG_SIZE = 128

def predict_nail_image(model, img_path, img_size, categories):
    img_array = cv2.imread(img_path, cv2.IMREAD_COLOR)
    img_resized = cv2.resize(img_array, (img_size, img_size)) / 255.0
    img_reshaped = np.reshape(img_resized, (1, img_size, img_size, 3))  # Reshape for the model
    prediction = loaded_model.predict(img_reshaped)
    predicted_class = np.argmax(prediction)
    predicted_category = categories[predicted_class]

    # Display the image and prediction
    plt.imshow(cv2.cvtColor(img_array, cv2.COLOR_BGR2RGB))
    plt.title(f"Predicted: {predicted_category}")
    plt.axis('off')
    plt.show()

    return predicted_category

# Test the model on a specific image
img_path = r"C:\Users\vaibh\OneDrive\Desktop\WhatsApp Image 2025-04-14 at 17.55.21_171e1112.jpg"  # Replace with the path to the image you want to test
predict_nail_image(loaded_model, img_path, IMG_SIZE, categories)
